In [0]:
from datetime import date, timedelta
from pyspark.sql.types import *

In [0]:
%run ../../utils/reader_utils

In [0]:
%run ../../utils/writer_utils

In [0]:
%run ../../utils/transform_utils

In [0]:
# Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
today = date.today()

ENV = configs.get("env", "dev")
INITIAL_RUN = configs.get('initial_run', False)
START_DATE = configs.get("start_date") or today - timedelta(days=1)
END_DATE = configs.get("end_date") or today + timedelta(days=1)
CATALOG = f"sl_{ENV}"

print(ENV, INITIAL_RUN, START_DATE, END_DATE, CATALOG)

In [0]:
# Constants
SOURCE_TABLE_CONF = {
    "table": "pyspark_scd2_orders",
    "schema": "silver",
    "timestamp_col": "_insert_update_ts"
}

SOURCE_TABLE_CONF_CITY_LOOKUP = {
    "table": "pyspark_static_location_lookup",
    "schema": "silver",
}

TARGET_TABLE_CONF = {
    "table": "pyspark_dim_location",
    "schema": "gold",
    "merge_keys": ["location_sk"],
    "primary_key": "location_sk"
}

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")

In [0]:
cities_df = (read_delta_table(SOURCE_TABLE_CONF, START_DATE, END_DATE)
             .select("origin_city", "destination_city")
             .filter((F.col("origin_city").isNotNull()) 
                     & (F.col("origin_city") != "")
                     & (F.col("destination_city").isNotNull())
                     & (F.col("destination_city") != "")))

In [0]:
cities_deduped_df = (cities_df
             .select("origin_city")
             .union(cities_df
                    .select("destination_city"))
             .dropDuplicates()
             .transform(add_normalized_str_col, "origin_city"))

In [0]:
static_location_lookup_df = (spark.read.table(f"""{SOURCE_TABLE_CONF_CITY_LOOKUP.get("schema")}.
                               {SOURCE_TABLE_CONF_CITY_LOOKUP.get("table")}""")
                             .transform(add_normalized_str_col, "city"))

In [0]:
location_df = (cities_deduped_df.alias("cty")
               .join(
                   F.broadcast(
                       static_location_lookup_df), 
                   "nrm_col", "left")
               .select(F.col("cty.origin_city"), 
                       *static_location_lookup_df.columns)
               .withColumn("missing_lookup", F.when(F.col("city").isNull(), F.lit("unknown")).otherwise(F.lit(None)))
               .withColumn("city", F.coalesce("city", "missing_lookup"))
               .drop("nrm_col", "origin_city", "missing_lookup")
               .dropDuplicates()
               )

In [0]:
final_location_df = (location_df
                     .transform(generate_sk, "location_sk", ["city"])
                     .transform(add_or_update_insert_update_ts))

In [0]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    target_table = f"{TARGET_TABLE_CONF.get('schema')}.{TARGET_TABLE_CONF.get('table')}"
    primary_key = TARGET_TABLE_CONF.get('primary_key')

    (final_location_df
     .writeTo(target_table)
     .using("delta")
     .createOrReplace())
    
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY AUTO")
    spark.sql(f"ALTER TABLE {target_table} ALTER COLUMN {primary_key} SET NOT NULL")
    spark.sql(f"ALTER TABLE {target_table} ADD CONSTRAINT {primary_key}_pk PRIMARY KEY ({primary_key})")
    
    dbutils.notebook.exit(f"Initial Table: {target_table} Setup Finished Successfully")

In [0]:
upsert_to_delta(final_location_df, TARGET_TABLE_CONF)